# 

In [185]:
#https://www.kaggle.com/competitions/latent-model-classification-aicc-round-0/overview


In [156]:
import os
from pathlib import Path
import random
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy.linalg as la
from sklearn.metrics import accuracy_score, confusion_matrix

SEED = 12345
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

INPUT_DIM = 100
EMBED_DIM = 10
LOGIT_DIM = 5
N = 5000
P_A = 0.5


In [157]:
OUT_DIR = Path("/kaggle/input/competitions/latent-model-classification-aicc-round-0")
TRAIN_CSV = OUT_DIR / "dataset.csv"
PEN_A_PTH = OUT_DIR / "modelA_penultimate.pth"
PEN_B_PTH = OUT_DIR / "modelB_penultimate.pth"
OUT_PRED_CSV = "/kaggle/working/predictions.csv"


In [158]:
df = pd.read_csv(TRAIN_CSV)
x_cols = [c for c in df.columns if c.startswith("x")]
X = df[x_cols].values.astype(np.float32)
class PenultimateNet(nn.Module):
    def __init__(self, in_dim=INPUT_DIM, hidden=64, out_dim=EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return self.net(x)

device = "cpu" # or "cuda"
penA = PenultimateNet().to(device)
penB = PenultimateNet().to(device)
penA.load_state_dict(torch.load(str(PEN_A_PTH), map_location=device))
penB.load_state_dict(torch.load(str(PEN_B_PTH), map_location=device))
penA.eval(); penB.eval()


PenultimateNet(
  (net): Sequential(
    (0): Linear(in_features=100, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [159]:
y_columns = np.array([c for c in df.columns if c.startswith("y")])
y = df[y_columns].values.astype(np.float32)


In [160]:
X.shape, y.shape

((5000, 100), (5000, 5))

In [161]:
A_features = penA(torch.from_numpy(X)).detach().numpy()
B_features = penB(torch.from_numpy(X)).detach().numpy()

In [162]:
A_features.shape, B_features.shape

((5000, 10), (5000, 10))

In [163]:
W_A = np.random.randn(10, 5)
W_B = np.random.randn(10, 5)


In [164]:
def probs(A, B, W_A,W_B, y):
    a_dist = np.sum((y - (A @ W_A))**2, axis = 1)
    b_dist = np.sum((y - (B@ W_B))** 2 , axis = 1)
    a_probs = np.exp(-a_dist)/(np.exp(-a_dist)+np.exp(-b_dist))
    b_probs = 1 - a_probs
    return a_probs, b_probs

In [165]:
def find_optimal_matrix(a_prob, b_prob, A, B, y):
    new_A = np.linalg.solve(A.T @ (A* a_prob[:, np.newaxis]), A.T @ (a_prob[:, np.newaxis]*y))
    new_B = np.linalg.solve(B.T @ (B * b_prob[:, np.newaxis]), B.T @ (b_prob[:, np.newaxis]*y))
    return new_A, new_B

In [166]:
epochs  = 50
for _ in range(epochs):
    a_prob, b_prob  = probs(A_features, B_features, W_A, W_B, y)
    W_A, W_B = find_optimal_matrix(a_prob, b_prob, A_features, B_features, y)


In [181]:
final_predictions = np.where(a_prob > 0.5, "A", "B")

In [182]:
submission = pd.DataFrame({
    'ID': df['ID'],
    'Source': final_predictions
})

In [183]:
submission.to_csv('submission.csv', index  =False)

In [184]:
submission


,ID,Source
0,1,B
1,2,B
2,3,B
3,4,B
4,5,A
...,...,...
4995,4996,B
4996,4997,A
4997,4998,B
4998,4999,B
